In [79]:
!python -V

Python 3.10.6


In [80]:
import pandas as pd

In [81]:
import pickle

In [82]:
import seaborn as sns
import matplotlib.pyplot as plt

In [83]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import root_mean_squared_error

In [84]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1717092736403, experiment_id='1', last_update_time=1717092736403, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [85]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [86]:
df_train = read_dataframe('data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('data/green_tripdata_2021-02.parquet')

In [87]:
df_train.columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge',
       'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge',
       'duration'],
      dtype='object')

In [88]:
len(df_train), len(df_val)

(73908, 61921)

In [89]:
df_train['PU_DO'] = df_train['PULocationID'].astype(str) + '_' + df_train['DOLocationID'].astype(str)

df_train['PU_DO'] = df_train['PULocationID'].astype(str) + '_' + df_train['DOLocationID'].astype(str)
df_val['PU_DO'] = df_val['PULocationID'].astype(str) + '_' + df_val['DOLocationID'].astype(str)

In [90]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [91]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [92]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

7.758715208946364

In [93]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [95]:
with mlflow.start_run():

    mlflow.set_tag("developer", "rui pinto")

    mlflow.log_param("train-data-path", "data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-path", "data/green_tripdata_2021-02.parquet")

    alpha = 0.01
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

In [96]:
import xgboost as xgb

In [97]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [98]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [99]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [100]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [20:25:13] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:11.79671                          
[1]	validation-rmse:11.40736                          
[2]	validation-rmse:11.04424                          
[3]	validation-rmse:10.70608                          
[4]	validation-rmse:10.39087                          
[5]	validation-rmse:10.09742                          
[6]	validation-rmse:9.82478                           
[7]	validation-rmse:9.57143                           
[8]	validation-rmse:9.33597                           
[9]	validation-rmse:9.11846                           
[10]	validation-rmse:8.91674                          
[11]	validation-rmse:8.73044                          
[12]	validation-rmse:8.55778                          
[13]	validation-rmse:8.39939                          
[14]	validation-rmse:8.25265                          
[15]	validation-rmse:8.11747                          
[16]	validation-rmse:7.99243                          
[17]	validation-rmse:7.87789                          
[18]	valid

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [20:26:04] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[2]	validation-rmse:6.93814                                                    
[3]	validation-rmse:6.79452                                                    
[4]	validation-rmse:6.73353                                                    
[5]	validation-rmse:6.70792                                                    
[6]	validation-rmse:6.68541                                                    
[7]	validation-rmse:6.67647                                                    
[8]	validation-rmse:6.67170                                                    
[9]	validation-rmse:6.66812                                                    
[10]	validation-rmse:6.66528                                                   
[11]	validation-rmse:6.66294                                                   
[12]	validation-rmse:6.66169                                                   
[13]	validation-rmse:6.65985                                                   
[14]	validation-rmse:6.65857            

...

[0]	validation-rmse:11.80973                                                   
[1]	validation-rmse:11.43230                                                   
[2]	validation-rmse:11.08027                                                   
[3]	validation-rmse:10.75157                                                   
[4]	validation-rmse:10.44547                                                   
[5]	validation-rmse:10.15877                                                   
[6]	validation-rmse:9.89369                                                    
[7]	validation-rmse:9.64599                                                    
[8]	validation-rmse:9.41780                                                    
[9]	validation-rmse:9.20370                                                    
[10]	validation-rmse:9.00587                                                   
[11]	validation-rmse:8.82368                                                   
[12]	validation-rmse:8.65292            

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [20:56:36] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:11.72647                                                   
[1]	validation-rmse:11.27697                                                   
[2]	validation-rmse:10.86315                                                   
[3]	validation-rmse:10.48187                                                   
[4]	validation-rmse:10.13088                                                   
[5]	validation-rmse:9.80906                                                    
[6]	validation-rmse:9.51396                                                    
[7]	validation-rmse:9.24284                                                    
[8]	validation-rmse:8.99465                                                    
[9]	validation-rmse:8.76759                                                    
[10]	validation-rmse:8.56066                                                   
[11]	validation-rmse:8.37280                                                   
[12]	validation-rmse:8.20231            

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [20:57:49] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:11.46022                                                    
[1]	validation-rmse:10.79608                                                    
[2]	validation-rmse:10.21328                                                    
[3]	validation-rmse:9.70509                                                     
[4]	validation-rmse:9.25852                                                     
[5]	validation-rmse:8.87174                                                     
[6]	validation-rmse:8.53425                                                     
[7]	validation-rmse:8.24329                                                     
[8]	validation-rmse:7.99304                                                     
[9]	validation-rmse:7.77677                                                     
[10]	validation-rmse:7.59040                                                    
[11]	validation-rmse:7.43038                                                    
[12]	validation-rmse:7.29355

In [101]:
mlflow.xgboost.autolog(disable=True)

In [102]:
with mlflow.start_run():

    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.0596230329181771,
        'max_depth': 55,
        'min_child_weight': 1.0733896735035067,
        'objective': 'reg:linear',
        'reg_alpha': 0.24434638248031015,
        'reg_lambda': 0.12315492972875541,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [21:09:41] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)


[0]	validation-rmse:11.72647
[1]	validation-rmse:11.27697
[2]	validation-rmse:10.86315
[3]	validation-rmse:10.48187
[4]	validation-rmse:10.13088
[5]	validation-rmse:9.80906
[6]	validation-rmse:9.51396
[7]	validation-rmse:9.24284
[8]	validation-rmse:8.99465
[9]	validation-rmse:8.76759
[10]	validation-rmse:8.56066
[11]	validation-rmse:8.37280
[12]	validation-rmse:8.20231
[13]	validation-rmse:8.04564
[14]	validation-rmse:7.90442
[15]	validation-rmse:7.77584
[16]	validation-rmse:7.65874
[17]	validation-rmse:7.55337
[18]	validation-rmse:7.45616
[19]	validation-rmse:7.36914
[20]	validation-rmse:7.28940
[21]	validation-rmse:7.21665
[22]	validation-rmse:7.15192
[23]	validation-rmse:7.09315
[24]	validation-rmse:7.03983
[25]	validation-rmse:6.99127
[26]	validation-rmse:6.94631
[27]	validation-rmse:6.90634
[28]	validation-rmse:6.87053
[29]	validation-rmse:6.83670
[30]	validation-rmse:6.80624
[31]	validation-rmse:6.77792
[32]	validation-rmse:6.75331
[33]	validation-rmse:6.73080
[34]	validation-rms

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [21:10:51] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.
  warnings.warn(smsg, UserWarning)


In [104]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "data/green_tripdata_2021-01.parquet")
        mlflow.log_param("valid-data-path", "data/green_tripdata_2021-02.parquet")
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)


2024/05/30 21:17:26 WARNING mlflow.utils.autologging_utils: You are using an unsupported version of sklearn. If you encounter errors during autologging, try upgrading / downgrading sklearn to a supported version, or try upgrading MLflow.
2024/05/30 21:17:26 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/_distutils_hack/__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils."
2024/05/30 21:17:26 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/ruifspinto/.pyenv/versions/3.1

AssertionError: /Users/ruifspinto/.pyenv/versions/3.10.6/lib/python3.10/distutils/core.py